# Notebook 3 — Feature Engineering

Build the training set for the binary classifier:
> *Given station S, target P, hour-of-day H — can S reach P within T minutes?*

## Plan

1. Annotate the OSM graph with effective edge speeds (LTA band where matched,
   else the OSM `maxspeed`, else our default 40 km/h).
2. Sample 500 target nodes per station from the road network → 2 000 pairs.
3. For each pair compute one base-traffic shortest path. Re-cost it at four
   timestamps (08:00, 13:00, 18:00, 23:00) using a time-of-day multiplier.
   This gives 8 000 labelled examples.
4. Compute features (distance, bearing, hour, path aggregates, rainfall) and
   ground-truth labels (1 if travel time ≤ 8 min).
5. Save a stratified 80/20 train/test split.

## Why this is 'synthetic' but defensible

We don't have access to SCDF dispatch records, so we use OSM + LTA traffic
as a proxy for actual emergency vehicle travel time — the same proxy any
external researcher would build. The time-of-day variation is simulated via
a multiplier on free-flow speeds rather than historical LTA snapshots; in
production we'd use the latter. The proposal calls this out explicitly.

In [1]:
# Auto-reload edits to src/ without restarting the kernel.
%load_ext autoreload
%autoreload 2

In [2]:
from __future__ import annotations

import logging
import random
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

import numpy as np
import pandas as pd
import networkx as nx
import osmnx as ox
from tqdm import tqdm

from src.config import DEMO_STATIONS, PROCESSED_DIR, DEFAULT_RESPONSE_THRESHOLD_MIN
from src.network import (
    load_road_network, assign_edge_speeds, shortest_path_travel_time,
)
from src.features import (
    haversine_km, manhattan_km, bearing_deg, cyclical_hour,
    aggregate_path, rainfall_along_path_mm, traffic_multiplier,
)

RNG_SEED = 0
TARGETS_PER_STATION = 500
TIMESTAMPS_HOURS = [8, 13, 18, 23]
THRESHOLD_MIN = DEFAULT_RESPONSE_THRESHOLD_MIN  # 8 minutes
DAY_OF_WEEK = 2  # Wednesday — held constant for the demo

rng = np.random.default_rng(RNG_SEED)
random.seed(RNG_SEED)

## 1. Load network + matched edges + rainfall, annotate speeds

In [3]:
G = load_road_network()
edges_traffic = pd.read_parquet(PROCESSED_DIR / 'edges_with_traffic.parquet')
rainfall = pd.read_parquet(PROCESSED_DIR / 'rainfall_latest.parquet')

print(f'Network:        {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
print(f'Edges w/ LTA:   {len(edges_traffic):,} ({len(edges_traffic)/G.number_of_edges():.1%})')
print(f'NEA stations:   {len(rainfall)}')

G = assign_edge_speeds(G, edges_traffic)
# Sanity check: distribution of base-traffic edge travel times.
times = [d['travel_time_min'] for _, _, d in G.edges(data=True) if 'travel_time_min' in d]
print(f'Edge travel-time stats (min): median={np.median(times):.3f}, p95={np.percentile(times,95):.3f}')

Network:        24,214 nodes, 46,082 edges
Edges w/ LTA:   16,138 (35.0%)
NEA stations:   75


Edge travel-time stats (min): median=0.123, p95=0.899


## 2. Largest connected component

Restrict sampling to the largest weakly connected component so every station
can actually reach every target. Otherwise shortest-path raises NoPath.

In [4]:
wcc = max(nx.weakly_connected_components(G), key=len)
G_main = G.subgraph(wcc).copy()
print(f'Largest WCC: {G_main.number_of_nodes():,} nodes ({len(wcc)/G.number_of_nodes():.1%} of graph)')

# Map each station to its nearest node in the main component.
station_nodes = {}
for s in DEMO_STATIONS:
    nid = ox.distance.nearest_nodes(G_main, X=s.lng, Y=s.lat)
    station_nodes[s.id] = nid
    print(f'  {s.name}: nearest node {nid}')

all_nodes = list(G_main.nodes())
print(f'\nSampling pool: {len(all_nodes):,} candidate target nodes')

Largest WCC: 24,214 nodes (100.0% of graph)
  Jurong Fire Station: nearest node 2051570271


  Bishan Fire Station: nearest node 5166391857
  Tampines Fire Station: nearest node 386362038


  Central Fire Station: nearest node 5765341406

Sampling pool: 24,214 candidate target nodes


## 3. Sample target nodes per station + compute base paths

One shortest-path computation per (station, target) at free-flow speeds.
The same path is then re-costed at each timestamp using a traffic multiplier.
This is a shortcut (in real traffic, optimal routes shift); documented.

In [5]:
pair_records = []  # base info per (station, target)

for station in DEMO_STATIONS:
    src_node = station_nodes[station.id]
    targets = rng.choice(all_nodes, size=TARGETS_PER_STATION, replace=False)
    failed = 0
    for tgt in tqdm(targets, desc=station.id, leave=False):
        if int(tgt) == int(src_node):
            failed += 1
            continue
        try:
            path, base_min = shortest_path_travel_time(G_main, src_node, int(tgt))
        except nx.NetworkXNoPath:
            failed += 1
            continue
        pair_records.append({
            'station_id': station.id,
            'station_lat': station.lat,
            'station_lng': station.lng,
            'src_node': src_node,
            'tgt_node': int(tgt),
            'tgt_lat': G_main.nodes[int(tgt)]['y'],
            'tgt_lng': G_main.nodes[int(tgt)]['x'],
            'path': path,
            'base_travel_min': base_min,
        })
    if failed:
        print(f'  {station.id}: {failed} pairs dropped (no path / same node)')

print(f'\nTotal (station, target) pairs: {len(pair_records):,}')

jurong:   0%|          | 0/500 [00:00<?, ?it/s]

jurong:   0%|          | 1/500 [00:00<02:58,  2.79it/s]

jurong:   1%|          | 3/500 [00:00<01:11,  6.94it/s]

jurong:   1%|          | 4/500 [00:01<02:20,  3.53it/s]

jurong:   1%|          | 5/500 [00:01<02:57,  2.80it/s]

...

central:  99%|█████████▊| 493/500 [01:29<00:00,  7.02it/s]

central:  99%|█████████▉| 494/500 [01:30<00:00,  7.12it/s]

central:  99%|█████████▉| 496/500 [01:30<00:00,  8.68it/s]

central: 100%|█████████▉| 498/500 [01:30<00:00,  9.51it/s]

  central: 6 pairs dropped (no path / same node)

Total (station, target) pairs: 1,973


## 4. Expand to (pair × timestamp) and compute features + labels

In [6]:
# Cache path aggregates and rainfall per pair so we don't recompute per hour.
aggregate_cache: dict[tuple[str, int], object] = {}
rainfall_cache: dict[tuple[str, int], float] = {}

rows = []
for rec in tqdm(pair_records, desc='features'):
    key = (rec['station_id'], rec['tgt_node'])
    if key not in aggregate_cache:
        aggregate_cache[key] = aggregate_path(G_main, rec['path'])
        rainfall_cache[key] = rainfall_along_path_mm(G_main, rec['path'], rainfall)
    pf = aggregate_cache[key]
    rain_mm = rainfall_cache[key]

    dist_km = haversine_km(rec['station_lat'], rec['station_lng'],
                           rec['tgt_lat'], rec['tgt_lng'])
    manh_km = manhattan_km(rec['station_lat'], rec['station_lng'],
                           rec['tgt_lat'], rec['tgt_lng'])
    brg = bearing_deg(rec['station_lat'], rec['station_lng'],
                      rec['tgt_lat'], rec['tgt_lng'])

    for hour in TIMESTAMPS_HOURS:
        mult = traffic_multiplier(hour)
        travel_min = rec['base_travel_min'] * mult
        hsin, hcos = cyclical_hour(hour)
        rows.append({
            # features
            'haversine_km': dist_km,
            'manhattan_km': manh_km,
            'bearing_deg': brg,
            'hour_sin': hsin,
            'hour_cos': hcos,
            'day_of_week': DAY_OF_WEEK,
            'num_segments': pf.num_segments,
            'avg_road_class': pf.avg_road_class,
            'expressway_share': pf.expressway_share,
            'avg_speed_band': pf.avg_speed_band,
            'matched_band_share': pf.matched_band_share,
            'path_length_km': pf.total_length_km,
            'rainfall_mm': rain_mm,
            'station_jurong': int(rec['station_id'] == 'jurong'),
            'station_bishan': int(rec['station_id'] == 'bishan'),
            'station_tampines': int(rec['station_id'] == 'tampines'),
            'station_central': int(rec['station_id'] == 'central'),
            # ground truth + metadata
            'true_travel_time_min': travel_min,
            'label': int(travel_min <= THRESHOLD_MIN),
            'station_id': rec['station_id'],
            'tgt_lat': rec['tgt_lat'],
            'tgt_lng': rec['tgt_lng'],
            'hour': hour,
        })

training_df = pd.DataFrame(rows)
print(f'Total examples: {len(training_df):,}')

features:   0%|          | 0/1973 [00:00<?, ?it/s]

features:   0%|          | 1/1973 [00:00<05:33,  5.91it/s]

features:   0%|          | 2/1973 [00:00<06:23,  5.14it/s]

features:   0%|          | 3/1973 [00:00<05:42,  5.75it/s]

features:   0%|          | 4/1973 [00:00<05:40,  5.79it/s]

...

features: 100%|█████████▉| 1971/1973 [04:59<00:00,  6.26it/s]

features: 100%|█████████▉| 1972/1973 [04:59<00:00,  6.79it/s]

features: 100%|██████████| 1973/1973 [04:59<00:00,  6.99it/s]

features: 100%|██████████| 1973/1973 [04:59<00:00,  6.58it/s]

Total examples: 7,892


## 5. Sanity check label balance + feature stats

In [7]:
print('Label distribution overall:')
print(training_df['label'].value_counts(normalize=True).rename('pct').to_frame().round(3))

print('\nLabel distribution by station × hour:')
print(training_df.groupby(['station_id', 'hour'])['label'].mean().unstack().round(2))

print('\nTravel time distribution (min):')
print(training_df['true_travel_time_min'].describe().round(2))

print('\nFeature summary:')
feature_cols = [
    'haversine_km', 'manhattan_km', 'bearing_deg',
    'hour_sin', 'hour_cos', 'day_of_week',
    'num_segments', 'avg_road_class', 'expressway_share',
    'avg_speed_band', 'matched_band_share', 'path_length_km', 'rainfall_mm',
    'station_jurong', 'station_bishan', 'station_tampines', 'station_central',
]
display(training_df[feature_cols].describe().T.round(3))

Label distribution overall:
         pct
label       
0      0.915
1      0.085

Label distribution by station × hour:
hour          8     13    18    23
station_id                        
bishan      0.07  0.15  0.05  0.20
central     0.04  0.12  0.03  0.14
jurong      0.03  0.08  0.03  0.09
tampines    0.05  0.09  0.04  0.12

Travel time distribution (min):
count    7892.00
mean       20.66
std        10.67
min         0.53
25%        12.87
50%        18.90
75%        26.50
max        76.83
Name: true_travel_time_min, dtype: float64

Feature summary:


,count,mean,std,min,25%,50%,75%,max
haversine_km,7892.0,11.541,6.674,0.085,6.430,10.769,15.389,38.552
manhattan_km,7892.0,14.189,7.773,0.116,8.129,13.265,19.454,48.553
bearing_deg,7892.0,181.892,105.754,0.215,80.475,209.533,272.712,359.885
hour_sin,7892.0,-0.163,0.667,-1.000,-0.444,-0.259,0.022,0.866
hour_cos,7892.0,-0.125,0.717,-0.966,-0.616,-0.250,0.241,0.966
day_of_week,7892.0,2.000,0.000,2.000,2.000,2.000,2.000,2.000
num_segments,7892.0,61.899,21.487,6.000,47.000,61.000,76.000,159.000
avg_road_class,7892.0,2.304,0.520,0.000,2.000,2.376,2.655,3.506
expressway_share,7892.0,0.629,0.273,0.000,0.537,0.725,0.822,0.947
avg_speed_band,7892.0,4.601,0.716,0.000,4.154,4.556,5.059,6.786


## 6. Train/test split (by target node, not by row)

Split at the **target-node level** so that the same geographic location never
appears in both train and test. This prevents spatial leakage — without it,
the model memorises specific locations rather than learning generalisable
distance/traffic patterns.

In [8]:
from sklearn.model_selection import train_test_split

# Split on unique target nodes so no location leaks between train and test.
training_df['target_key'] = (
    training_df['station_id'] + '_'
    + training_df['tgt_lat'].astype(str) + '_'
    + training_df['tgt_lng'].astype(str)
)

unique_keys = list(training_df['target_key'].unique())
train_keys, test_keys = train_test_split(
    unique_keys,
    test_size=0.2,
    random_state=RNG_SEED,
)
train_keys_set = set(train_keys)
test_keys_set = set(test_keys)

train_df = training_df[training_df['target_key'].isin(train_keys_set)].drop(columns=['target_key'])
test_df = training_df[training_df['target_key'].isin(test_keys_set)].drop(columns=['target_key'])
training_df = training_df.drop(columns=['target_key'])

print(f'Unique target nodes — train: {len(train_keys):,}, test: {len(test_keys):,}')
print(f'Train: {len(train_df):,}  ({train_df["label"].mean():.1%} positive)')
print(f'Test:  {len(test_df):,}  ({test_df["label"].mean():.1%} positive)')

training_df.to_parquet(PROCESSED_DIR / 'training_data.parquet', index=False)
train_df.to_parquet(PROCESSED_DIR / 'training_data_train.parquet', index=False)
test_df.to_parquet(PROCESSED_DIR / 'training_data_test.parquet', index=False)
print(f'\nSaved to {PROCESSED_DIR}/training_data{{,_train,_test}}.parquet')

Unique target nodes — train: 1,578, test: 395
Train: 6,312  (8.8% positive)
Test:  1,580  (7.0% positive)

Saved to C:\Users\wenji\Downloads\scdf-coverage-extracted\scdf-coverage\data\processed/training_data{,_train,_test}.parquet
